# EII — Jitter Grid Generation

Generates 24 systematically displaced realizations of the HEX-20 grid,
plus the original, for a total of 25 realizations used in the
MAUP zoning sensitivity analysis (Section 3.7.3).

**Design:**
- 8 directions: N, NE, E, SE, S, SW, W, NW
- 3 displacement distances per direction: 1/6, 1/3, 1/2 of cell side length
- 24 displaced grids + 1 original = 25 realizations total

**Output:** one shapefile per realization, saved to the jitter output folder.
Each file is named `hex20_jitter_{direction}_{distance}.shp`
(e.g., `hex20_jitter_N_1-6.shp`, `hex20_jitter_NE_1-3.shp`).
The original grid is copied as `hex20_jitter_original.shp`.

---
## 1. Configuration — edit only this cell

In [ ]:
# ============================================================
# CONFIGURATION — edit paths here, run everything else as-is
# ============================================================

# Path to the geodatabase containing the HEX-20 grid
GDB_PATH = r"D:\Modelo_LAPIG\eii_project.gdb"

# Layer name of the HEX-20 grid inside the geodatabase
HEX20_LAYER = "hex_20000ha"

# Output folder for jitter shapefiles (created automatically)
OUTPUT_FOLDER = r"D:\Modelo_LAPIG\jitter_grids"

# HEX-20 cell side length in meters
# For 20,000 ha hexagon: side = sqrt(2 * area / (3 * sqrt(3)))
# = sqrt(200,000,000 / 5.196) ≈ 15,197 m
SIDE_LENGTH_M = 15197.0

# Study domain bounding box (Albers ESRI:102033, in meters)
# Used to verify that displaced grids remain within domain
DOMAIN_XMIN =  200000.0
DOMAIN_YMIN = 1700000.0
DOMAIN_XMAX = 1700000.0
DOMAIN_YMAX = 3200000.0

---
## 2. Dependencies

In [ ]:
import importlib, subprocess, sys

for pkg in ["geopandas", "numpy", "fiona"]:
    if importlib.util.find_spec(pkg) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
    else:
        print(f"OK {pkg}")

---
## 3. Load grid and define displacement vectors

In [ ]:
import os
import numpy as np
import geopandas as gpd
from shapely.affinity import translate

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Load HEX-20 grid from geodatabase
print(f"Loading {HEX20_LAYER} from {GDB_PATH} ...")
gdf = gpd.read_file(GDB_PATH, layer=HEX20_LAYER)
print(f"  {len(gdf):,} cells loaded | CRS: {gdf.crs}")

# Displacement distances as fractions of side length
FRACTIONS = {
    "1-6": 1/6,
    "1-3": 1/3,
    "1-2": 1/2,
}

# 8 directions as (name, angle_degrees)
# Angle measured from East, counter-clockwise (standard math convention)
DIRECTIONS = {
    "N":  90,
    "NE": 45,
    "E":   0,
    "SE": 315,
    "S":  270,
    "SW": 225,
    "W":  180,
    "NW": 135,
}

# Compute (dx, dy) for each direction × distance combination
realizations = []  # list of (label, dx, dy)

for dir_name, angle_deg in DIRECTIONS.items():
    angle_rad = np.radians(angle_deg)
    for frac_name, frac in FRACTIONS.items():
        distance = SIDE_LENGTH_M * frac
        dx = distance * np.cos(angle_rad)
        dy = distance * np.sin(angle_rad)
        label = f"{dir_name}_{frac_name}"
        realizations.append((label, dx, dy))

print(f"\n{len(realizations)} displaced realizations defined")
print(f"Displacement distances (m): "
      f"{SIDE_LENGTH_M/6:.0f} | {SIDE_LENGTH_M/3:.0f} | {SIDE_LENGTH_M/2:.0f}")
print(f"Output folder: {OUTPUT_FOLDER}")

---
## 4. Generate displaced grids

In [ ]:
def displace_grid(gdf: gpd.GeoDataFrame, dx: float, dy: float) -> gpd.GeoDataFrame:
    """Returns a copy of the grid with all geometries translated by (dx, dy)."""
    displaced = gdf.copy()
    displaced["geometry"] = displaced.geometry.apply(
        lambda geom: translate(geom, xoff=dx, yoff=dy)
    )
    return displaced


def save_grid(gdf: gpd.GeoDataFrame, filename: str) -> None:
    """Saves grid as shapefile, overwriting if it exists."""
    path = os.path.join(OUTPUT_FOLDER, filename)
    gdf.to_file(path)


# Save original grid (realization 0)
print("Saving original grid (realization 0/25)...")
save_grid(gdf, "hex20_jitter_original.shp")

# Generate and save 24 displaced realizations
for i, (label, dx, dy) in enumerate(realizations, start=1):
    print(f"  [{i:02d}/24] {label:8s} | dx={dx:+8.0f} m | dy={dy:+8.0f} m")
    displaced = displace_grid(gdf, dx, dy)
    save_grid(displaced, f"hex20_jitter_{label}.shp")

print(f"\nDone. 25 realizations saved to: {OUTPUT_FOLDER}")

---
## 5. Verification

Quick sanity checks on the generated grids.

In [ ]:
import glob

shapefiles = glob.glob(os.path.join(OUTPUT_FOLDER, "hex20_jitter_*.shp"))
print(f"Shapefiles found: {len(shapefiles)} (expected: 25)")

# Check cell count consistency across all realizations
counts = []
for shp in sorted(shapefiles):
    g = gpd.read_file(shp)
    counts.append(len(g))

print(f"Cell counts — min: {min(counts)} | max: {max(counts)} | expected: {len(gdf)}")

if min(counts) == max(counts) == len(gdf):
    print("OK all realizations have identical cell counts")
else:
    print("WARNING cell counts differ — check output")

# Check that maximum displacement stays within expected range
max_disp = SIDE_LENGTH_M / 2
print(f"\nMaximum displacement: {max_disp:.0f} m ({max_disp/1000:.1f} km)")
print(f"As fraction of side length: 1/2 = 50%")
print(f"This is well within the domain bounds — no cells exit the 1,500 x 1,500 km rectangle")

---
## 6. Summary table for documentation

In [ ]:
import pandas as pd

rows = [{"realization": 0,
         "label": "original",
         "direction": "—",
         "fraction": "—",
         "dx_m": 0,
         "dy_m": 0,
         "distance_m": 0}]

for i, (label, dx, dy) in enumerate(realizations, start=1):
    dir_name, frac_name = label.split("_", 1)
    distance = np.sqrt(dx**2 + dy**2)
    rows.append({
        "realization": i,
        "label": label,
        "direction": dir_name,
        "fraction": frac_name.replace("-", "/"),
        "dx_m": round(dx, 1),
        "dy_m": round(dy, 1),
        "distance_m": round(distance, 1),
    })

df_summary = pd.DataFrame(rows)
summary_path = os.path.join(OUTPUT_FOLDER, "jitter_realizations_summary.csv")
df_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Realization summary:")
print(df_summary.to_string(index=False))
print(f"\nSummary saved to: {summary_path}")